## Email Sender Test Notebook

SMTP 연동 및 발송 로직 테스트

### 목차
1. [Setup & Imports](#setup)
2. [환경 변수 확인](#env-check)
3. [EmailSender 초기화](#init)
4. [단일 이메일 발송 테스트](#single-email)
5. [HTML + Plain Text 이메일](#multipart)
6. [완전한 다이제스트 이메일 발송](#full-digest)
7. [배치 이메일 발송](#batch-email)
8. [에러 핸들링 테스트](#error-handling)
9. [재시도 로직 테스트](#retry-test)
10. [성능 테스트](#performance)


#### 필요한 환경 변수 (.env 파일에 추가)

```bash
# SMTP Configuration
SMTP_HOST=smtp.gmail.com
SMTP_PORT=587
SMTP_USER=your-email@gmail.com
SMTP_PASSWORD=your-app-password  # ⚠️ Gmail 앱 비밀번호 필요
SMTP_FROM_EMAIL=your-email@gmail.com
SMTP_FROM_NAME=research-curator
```

### Gmail 앱 비밀번호 생성 방법

1. Google 계정 → 보안 설정
2. 2단계 인증 활성화 (필수)
3. 앱 비밀번호 생성
   - https://myaccount.google.com/apppasswords
   - "메일" 선택
   - 16자리 비밀번호 생성
   - `.env` 파일의 `SMTP_PASSWORD`에 입력 (공백 제거)

### 테스트 대상
- `src/app/email/sender.py` - EmailSender 클래스
- `src/app/email/builder.py` - EmailBuilder 클래스 (통합 테스트용)

In [1]:
import sys
import asyncio
import time
from pathlib import Path
from datetime import datetime, UTC
from uuid import uuid4
from dotenv import load_dotenv
import os

# Add src to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

# Load environment variables
load_dotenv(project_root / ".env")

from app.email.sender import EmailSender, send_email, send_batch_emails
from app.email.builder import EmailBuilder
from app.db.models import CollectedArticle

print("✅ Imports successful")
print(f"📁 Project root: {project_root}")

✅ Imports successful
📁 Project root: /mnt/d/project/research-curator


In [2]:
# 환경변수 확인
print("📧 SMTP Configuration Check:")
print("=" * 80)

# Check required environment variables
required_vars = [
    "SMTP_HOST",
    "SMTP_PORT",
    "SMTP_USER",
    "SMTP_PASSWORD",
    "SMTP_FROM_EMAIL",
    "SMTP_FROM_NAME",
]

all_set = True
for var in required_vars:
    value = os.getenv(var)
    if value:
        # Mask password
        if "PASSWORD" in var:
            display_value = "*" * 16 if len(value) > 16 else "*" * len(value)
        else:
            display_value = value
        print(f"✅ {var:<20}: {display_value}")
    else:
        print(f"❌ {var:<20}: NOT SET")
        all_set = False

print("\n" + "=" * 80)
if all_set:
    print("✅ All required environment variables are set!")
    print("\n⚠️  Make sure you're using Gmail App Password, not regular password!")
else:
    print("❌ Some environment variables are missing!")
    print("Please check .env file and add missing variables.")

📧 SMTP Configuration Check:
✅ SMTP_HOST           : smtp.gmail.com
✅ SMTP_PORT           : 587
✅ SMTP_USER           : sguys99@gmail.com
✅ SMTP_PASSWORD       : ****************
✅ SMTP_FROM_EMAIL     : sguys99@gmail.com
✅ SMTP_FROM_NAME      : research-curator

✅ All required environment variables are set!

⚠️  Make sure you're using Gmail App Password, not regular password!


### 1. EmailSender 초기화 <a name="init"></a>

In [3]:
print("🔧 Initializing EmailSender...")
print("=" * 80)

try:
    # Initialize with environment variables
    sender = EmailSender()
    
    print("✅ EmailSender initialized successfully!")
    print(f"\n📋 Configuration:")
    print(f"   SMTP Host: {sender.smtp_host}")
    print(f"   SMTP Port: {sender.smtp_port}")
    print(f"   SMTP User: {sender.smtp_user}")
    print(f"   From Email: {sender.from_email}")
    print(f"   From Name: {sender.from_name}")
    print(f"   Password Set: {'✅ Yes' if sender.smtp_password else '❌ No'}")
    
except ValueError as e:
    print(f"❌ Initialization failed: {e}")
    print("\nPlease check your .env file and ensure all SMTP variables are set.")

🔧 Initializing EmailSender...
✅ EmailSender initialized successfully!

📋 Configuration:
   SMTP Host: smtp.gmail.com
   SMTP Port: 587
   SMTP User: sguys99@gmail.com
   From Email: sguys99@gmail.com
   From Name: research-curator
   Password Set: ✅ Yes


### 2. 단일 이메일 발송 테스트 <a name="single-email"></a>

간단한 HTML 이메일을 발송합니다.

In [4]:
print("📤 Test 1: Simple HTML Email")
print("=" * 80)

# Target email
recipient = "sguys99@gmail.com"

# Simple HTML content - using f-string instead of .format() to avoid CSS {} conflicts
current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <style>
        body {{ font-family: Arial, sans-serif; line-height: 1.6; color: #333; }}
        .container {{ max-width: 600px; margin: 0 auto; padding: 20px; }}
        .header {{ background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                  color: white; padding: 30px; text-align: center; border-radius: 10px; }}
        .content {{ background: #f9f9f9; padding: 30px; margin-top: 20px;
                   border-radius: 10px; border: 1px solid #ddd; }}
        .footer {{ text-align: center; margin-top: 20px; color: #666; font-size: 12px; }}
        .emoji {{ font-size: 24px; }}
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <h1><span class="emoji">🧪</span> Email Sender Test</h1>
            <p>Testing EmailSender functionality</p>
        </div>

        <div class="content">
            <h2>Hello from Research Curator! 👋</h2>
            <p>This is a <strong>test email</strong> sent from the Research Curator email system.</p>

            <h3>Test Details:</h3>
            <ul>
                <li><strong>Test Type:</strong> Single Email Send</li>
                <li><strong>Sender:</strong> EmailSender class</li>
                <li><strong>Time:</strong> {current_time}</li>
                <li><strong>Purpose:</strong> Verify SMTP connection and HTML rendering</li>
            </ul>

            <p>✅ If you're reading this, the email was successfully delivered!</p>
        </div>

        <div class="footer">
            <p>Sent by Research Curator Test Suite</p>
            <p>Day 6 - Checkpoint 2: SMTP Integration</p>
        </div>
    </div>
</body>
</html>
"""

print(f"📧 Recipient: {recipient}")
print(f"📝 Subject: Research Curator - Test Email")
print(f"📏 HTML Size: {len(html_content)} characters")
print("\n⏳ Sending email...")

try:
    # Send email
    success = await sender.send_email(
        to_email=recipient,
        subject="🧪 Research Curator - Test Email",
        html_content=html_content
    )

    if success:
        print("\n✅ Email sent successfully!")
        print(f"\n📬 Please check {recipient} inbox")
        print("\n💡 Tips:")
        print("   - Check spam folder if not in inbox")
        print("   - Look for subject: 🧪 Research Curator - Test Email")
    else:
        print("\n❌ Email sending failed")

except Exception as e:
    print(f"\n❌ Error sending email: {e}")
    print("\n🔍 Common issues:")
    print("   - Wrong Gmail App Password")
    print("   - 2-Step Verification not enabled")
    print("   - Using regular password instead of App Password")
    print("   - SMTP settings incorrect")

📤 Test 1: Simple HTML Email
📧 Recipient: sguys99@gmail.com
📝 Subject: Research Curator - Test Email
📏 HTML Size: 1717 characters

⏳ Sending email...

✅ Email sent successfully!

📬 Please check sguys99@gmail.com inbox

💡 Tips:
   - Check spam folder if not in inbox
   - Look for subject: 🧪 Research Curator - Test Email


### 3. HTML + Plain Text 이메일 <a name="multipart"></a>

Multipart 이메일 (HTML + Plain Text fallback)

In [5]:
print("📤 Test 2: Multipart Email (HTML + Plain Text)")
print("=" * 80)

# Plain text version
current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
text_content = f"""
Research Curator - Multipart Email Test
========================================

Hello from Research Curator!

This is a test email with both HTML and plain text versions.

Test Details:
- Test Type: Multipart Email
- Time: {current_time}
- Purpose: Test email client compatibility

If you see this text, your email client is using the plain text version.
Modern email clients will show the HTML version instead.

---
Sent by Research Curator Test Suite
Day 6 - Checkpoint 2
"""

# HTML version
html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <style>
        body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
               background: #f4f4f4; padding: 20px; }}
        .card {{ background: white; max-width: 600px; margin: 0 auto;
                border-radius: 15px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); overflow: hidden; }}
        .banner {{ background: linear-gradient(90deg, #00c6ff 0%, #0072ff 100%);
                  color: white; padding: 40px; text-align: center; }}
        .body {{ padding: 40px; }}
        .badge {{ display: inline-block; background: #00c6ff; color: white;
                 padding: 5px 15px; border-radius: 20px; font-size: 12px;
                 font-weight: bold; margin: 5px; }}
        .footer {{ background: #f9f9f9; padding: 20px; text-align: center;
                  border-top: 1px solid #ddd; }}
    </style>
</head>
<body>
    <div class="card">
        <div class="banner">
            <h1 style="margin: 0; font-size: 32px;">📧 Multipart Email Test</h1>
            <p style="margin: 10px 0 0 0; opacity: 0.9;">HTML + Plain Text Version</p>
        </div>

        <div class="body">
            <h2 style="color: #333;">Hello from Research Curator! 👋</h2>
            <p style="color: #666; line-height: 1.8;">
                This email contains <strong>both HTML and plain text</strong> versions.
                Your email client automatically chose to display the HTML version.
            </p>

            <div style="background: #f0f8ff; padding: 20px; border-radius: 10px;
                        border-left: 4px solid #0072ff; margin: 20px 0;">
                <h3 style="margin: 0 0 10px 0; color: #0072ff;">Why Multipart?</h3>
                <p style="margin: 0; color: #666;">
                    Multipart emails ensure compatibility with all email clients.
                    If HTML is not supported, the plain text version is displayed.
                </p>
            </div>

            <p style="color: #666;">
                <span class="badge">HTML Version</span>
                <span class="badge">Styled</span>
                <span class="badge">Modern</span>
            </p>

            <p style="color: #666;">✅ Email sent at: <strong>{current_time}</strong></p>
        </div>

        <div class="footer">
            <p style="margin: 0; color: #999; font-size: 14px;">
                Research Curator Test Suite | Day 6 Checkpoint 2
            </p>
        </div>
    </div>
</body>
</html>
"""

print(f"📧 Recipient: {recipient}")
print(f"📝 Subject: Research Curator - Multipart Email Test")
print(f"📏 Plain Text Size: {len(text_content)} characters")
print(f"📏 HTML Size: {len(html_content)} characters")
print("\n⏳ Sending multipart email...")

try:
    success = await sender.send_email(
        to_email=recipient,
        subject="📧 Research Curator - Multipart Email Test",
        html_content=html_content,
        text_content=text_content  # Fallback for non-HTML clients
    )

    if success:
        print("\n✅ Multipart email sent successfully!")
        print("\n📱 Check your email client:")
        print("   - Modern clients (Gmail, Outlook) will show HTML version")
        print("   - Text-only clients will show plain text version")
    else:
        print("\n❌ Email sending failed")

except Exception as e:
    print(f"\n❌ Error: {e}")

📤 Test 2: Multipart Email (HTML + Plain Text)
📧 Recipient: sguys99@gmail.com
📝 Subject: Research Curator - Multipart Email Test
📏 Plain Text Size: 482 characters
📏 HTML Size: 2512 characters

⏳ Sending multipart email...

✅ Multipart email sent successfully!

📱 Check your email client:
   - Modern clients (Gmail, Outlook) will show HTML version
   - Text-only clients will show plain text version


### 4. 완전한 다이제스트 이메일 발송 <a name="full-digest"></a>

EmailBuilder와 통합하여 실제 일일 다이제스트 이메일 발송

In [6]:
print("📤 Test 3: Full Daily Digest Email")
print("=" * 80)

# Create sample articles
sample_articles = [
    CollectedArticle(
        id=str(uuid4()),
        title="Attention Is All You Need",
        content="We propose a new simple network architecture, the Transformer...",
        summary="트랜스포머 아키텍처를 제안하는 획기적인 논문. 어텐션 메커니즘만으로 시퀀스 모델링을 수행합니다.",
        source_url="https://arxiv.org/abs/1706.03762",
        source_type="paper",
        importance_score=0.95,
        collected_at=datetime.now(UTC),
        article_metadata={
            "authors": ["Ashish Vaswani", "Noam Shazeer", "Niki Parmar"],
            "citations": 50000,
        },
    ),
    CollectedArticle(
        id=str(uuid4()),
        title="OpenAI Announces GPT-5",
        content="OpenAI today announced GPT-5, the latest iteration...",
        summary="OpenAI가 GPT-5를 발표했습니다. 이전 모델보다 10배 향상된 성능을 보입니다.",
        source_url="https://techcrunch.com/gpt5",
        source_type="news",
        importance_score=0.88,
        collected_at=datetime.now(UTC),
        article_metadata={"source": "TechCrunch"},
    ),
    CollectedArticle(
        id=str(uuid4()),
        title="Stanford AI Index Report 2024",
        content="The AI Index 2024 Annual Report tracks, collates...",
        summary="스탠포드 AI 인덱스 2024 리포트가 발표되었습니다. AI 산업 전반의 동향을 분석합니다.",
        source_url="https://aiindex.stanford.edu/report/",
        source_type="report",
        importance_score=0.82,
        collected_at=datetime.now(UTC),
        article_metadata={"organization": "Stanford HAI"},
    ),
]

print(f"📚 Prepared {len(sample_articles)} sample articles")
print("\n🏗️  Building email with EmailBuilder...")

# Build email content
builder = EmailBuilder()
html_digest = builder.build_daily_digest(
    user_name="연구자",
    user_email=recipient,
    articles=sample_articles,
    daily_limit=5,
)

print(f"✅ Email built successfully")
print(f"📏 HTML Size: {len(html_digest):,} characters ({len(html_digest)/1024:.1f} KB)")
print(f"\n📧 Recipient: {recipient}")
print(f"📝 Subject: 🔬 오늘의 AI 연구 다이제스트")
print("\n⏳ Sending daily digest email...")

try:
    success = await sender.send_email(
        to_email=recipient,
        subject=f"🔬 오늘의 AI 연구 다이제스트 - {datetime.now().strftime('%Y년 %m월 %d일')}",
        html_content=html_digest
    )
    
    if success:
        print("\n✅ Daily digest email sent successfully!")
        print("\n📬 Check your inbox for:")
        print("   📚 Papers section (Attention Is All You Need)")
        print("   📰 News section (GPT-5 announcement)")
        print("   📊 Reports section (Stanford AI Index)")
        print("\n🎨 Email Features:")
        print("   ✓ Responsive design (mobile + desktop)")
        print("   ✓ Importance stars (⭐⭐⭐)")
        print("   ✓ Personalized greeting")
        print("   ✓ Footer links (Settings, Feedback, Unsubscribe)")
    else:
        print("\n❌ Email sending failed")
        
except Exception as e:
    print(f"\n❌ Error: {e}")

📤 Test 3: Full Daily Digest Email
📚 Prepared 3 sample articles

🏗️  Building email with EmailBuilder...
✅ Email built successfully
📏 HTML Size: 13,722 characters (13.4 KB)

📧 Recipient: sguys99@gmail.com
📝 Subject: 🔬 오늘의 AI 연구 다이제스트

⏳ Sending daily digest email...

✅ Daily digest email sent successfully!

📬 Check your inbox for:
   📚 Papers section (Attention Is All You Need)
   📰 News section (GPT-5 announcement)
   📊 Reports section (Stanford AI Index)

🎨 Email Features:
   ✓ Responsive design (mobile + desktop)
   ✓ Importance stars (⭐⭐⭐)
   ✓ Personalized greeting
   ✓ Footer links (Settings, Feedback, Unsubscribe)


### 5. 배치 이메일 발송 <a name="batch-email"></a>

여러 사용자에게 동시에 이메일 발송 (테스트용으로 같은 주소에 여러 번)

In [7]:
print("📤 Test 4: Batch Email Sending")
print("=" * 80)

# Prepare batch recipients (테스트용으로 같은 주소에 3개 발송)
time1 = datetime.now().strftime("%H:%M:%S")
time2 = datetime.now().strftime("%H:%M:%S")
time3 = datetime.now().strftime("%H:%M:%S")

recipients = [
    {
        "to_email": recipient,
        "subject": "🧪 Research Curator - Batch Test #1",
        "html_content": f"""
        <html>
        <body style="font-family: Arial; padding: 20px;">
            <h1 style="color: #667eea;">📧 Batch Email Test #1</h1>
            <p>This is the <strong>first email</strong> in the batch test.</p>
            <p>Time: {time1}</p>
        </body>
        </html>
        """,
    },
    {
        "to_email": recipient,
        "subject": "🧪 Research Curator - Batch Test #2",
        "html_content": f"""
        <html>
        <body style="font-family: Arial; padding: 20px;">
            <h1 style="color: #764ba2;">📧 Batch Email Test #2</h1>
            <p>This is the <strong>second email</strong> in the batch test.</p>
            <p>Time: {time2}</p>
        </body>
        </html>
        """,
    },
    {
        "to_email": recipient,
        "subject": "🧪 Research Curator - Batch Test #3",
        "html_content": f"""
        <html>
        <body style="font-family: Arial; padding: 20px;">
            <h1 style="color: #00c6ff;">📧 Batch Email Test #3</h1>
            <p>This is the <strong>third email</strong> in the batch test.</p>
            <p>Time: {time3}</p>
            <p>✅ All batch emails sent successfully!</p>
        </body>
        </html>
        """,
    },
]

print(f"📧 Batch size: {len(recipients)} emails")
print(f"📬 All sending to: {recipient}")
print("\n⏳ Sending batch emails...")

start_time = time.time()

try:
    result = await sender.send_batch_emails(
        recipients=recipients,
        max_failures=2  # Stop if 2 failures occur
    )

    elapsed = time.time() - start_time

    print(f"\n✅ Batch sending completed in {elapsed:.2f}s")
    print("\n📊 Results:")
    print(f"   ✅ Successful: {result['success_count']}")
    print(f"   ❌ Failed: {result['failure_count']}")
    print(f"   ⏱️  Avg time per email: {elapsed/len(recipients):.2f}s")

    if result['failed_emails']:
        print("\n❌ Failed Emails:")
        for failed in result['failed_emails']:
            print(f"   - {failed['email']}: {failed['error']}")

    if result['success_count'] == len(recipients):
        print("\n🎉 All emails sent successfully!")
        print(f"\n📬 Check {recipient} inbox for 3 emails")

except Exception as e:
    print(f"\n❌ Batch sending error: {e}")

📤 Test 4: Batch Email Sending
📧 Batch size: 3 emails
📬 All sending to: sguys99@gmail.com

⏳ Sending batch emails...

✅ Batch sending completed in 8.42s

📊 Results:
   ✅ Successful: 3
   ❌ Failed: 0
   ⏱️  Avg time per email: 2.81s

🎉 All emails sent successfully!

📬 Check sguys99@gmail.com inbox for 3 emails


### 6. 에러 핸들링 테스트 <a name="error-handling"></a>

In [8]:
print("🧪 Test 5: Error Handling")
print("=" * 80)

# Test 1: Invalid email address
print("\n[Test 5-1] Invalid email address...")
try:
    await sender.send_email(
        to_email="invalid-email",  # Invalid format
        subject="Test",
        html_content="<p>Test</p>"
    )
    print("❌ Should have raised an error")
except Exception as e:
    print(f"✅ Caught expected error: {type(e).__name__}")
    print(f"   Message: {str(e)[:100]}")

# Test 2: Empty content
print("\n[Test 5-2] Empty content...")
try:
    await sender.send_email(
        to_email=recipient,
        subject="Empty Content Test",
        html_content=""  # Empty HTML
    )
    # Empty content is actually valid, so this should succeed
    print("✅ Empty content handled (email sent with empty body)")
except Exception as e:
    print(f"⚠️  Error with empty content: {e}")

# Test 3: Very long subject
print("\n[Test 5-3] Very long subject line...")
long_subject = "A" * 500  # 500 characters
try:
    await sender.send_email(
        to_email=recipient,
        subject=long_subject,
        html_content="<p>Test with very long subject</p>"
    )
    print("✅ Long subject handled successfully")
except Exception as e:
    print(f"⚠️  Long subject error: {type(e).__name__}")

print("\n✅ Error handling tests complete")

🧪 Test 5: Error Handling

[Test 5-1] Invalid email address...


Failed to send email to invalid-email: [SMTPRecipientRefused(553, '5.1.3 The recipient address <invalid-email> is not a valid RFC 5321 address.\n5.1.3 For more information, go to\n5.1.3  https://support.google.com/a/answer/3221692 and review RFC 5321\n5.1.3 specifications. d2e1a72fcca58-7fe13d853a9sm2738022b3a.42 - gsmtp', 'invalid-email')]
Failed to send email to invalid-email: [SMTPRecipientRefused(553, '5.1.3 The recipient address <invalid-email> is not a valid RFC 5321 address.\n5.1.3 For more information, go to\n5.1.3  https://support.google.com/a/answer/3221692 and review RFC 5321\n5.1.3 specifications. d9443c01a7336-2a2d161272asm27075135ad.55 - gsmtp', 'invalid-email')]
Failed to send email to invalid-email: [SMTPRecipientRefused(553, '5.1.3 The recipient address <invalid-email> is not a valid RFC 5321 address.\n5.1.3 For more information, go to\n5.1.3  https://support.google.com/a/answer/3221692 and review RFC 5321\n5.1.3 specifications. 98e67ed59e1d1-34e70dccd14sm2625285a91.16

✅ Caught expected error: SMTPRecipientsRefused
   Message: [SMTPRecipientRefused(553, '5.1.3 The recipient address <invalid-email> is not a valid RFC 5321 addr

[Test 5-2] Empty content...
✅ Empty content handled (email sent with empty body)

[Test 5-3] Very long subject line...
✅ Long subject handled successfully

✅ Error handling tests complete


### 7. 재시도 로직 테스트 <a name="retry-test"></a>

Tenacity의 재시도 메커니즘 검증

In [9]:
print("🔄 Test 6: Retry Logic")
print("=" * 80)

print("""
EmailSender.send_email() has built-in retry logic:
- Max retries: 3 attempts
- Backoff strategy: Exponential (2s → 4s → 8s)
- Library: tenacity

Testing with a valid email to observe retry behavior in logs...
""")

print("⏳ Sending email (watch for retry logs if connection fails)...")

try:
    success = await sender.send_email(
        to_email=recipient,
        subject="🔄 Research Curator - Retry Test",
        html_content="""
        <html>
        <body style="font-family: Arial; padding: 20px;">
            <h1>🔄 Retry Logic Test</h1>
            <p>This email tests the automatic retry mechanism.</p>
            <ul>
                <li>Max retries: 3</li>
                <li>Wait times: 2s, 4s, 8s (exponential backoff)</li>
                <li>Library: tenacity</li>
            </ul>
            <p>✅ If you received this email, the retry logic worked!</p>
        </body>
        </html>
        """
    )
    
    if success:
        print("\n✅ Email sent (potentially after retries)")
        print("\n📝 Note:")
        print("   - If connection is stable, email sends on first try")
        print("   - If connection fails temporarily, auto-retry kicks in")
        print("   - Check console logs for retry messages")
    else:
        print("\n❌ Email failed after all retries")
        
except Exception as e:
    print(f"\n❌ Final error after retries: {e}")
    print("\nThis means all 3 retry attempts failed.")

🔄 Test 6: Retry Logic

EmailSender.send_email() has built-in retry logic:
- Max retries: 3 attempts
- Backoff strategy: Exponential (2s → 4s → 8s)
- Library: tenacity

Testing with a valid email to observe retry behavior in logs...

⏳ Sending email (watch for retry logs if connection fails)...

✅ Email sent (potentially after retries)

📝 Note:
   - If connection is stable, email sends on first try
   - If connection fails temporarily, auto-retry kicks in
   - Check console logs for retry messages


## 8. 성능 테스트 <a name="performance"></a>

In [10]:
print("⚡ Test 7: Performance Benchmark")
print("=" * 80)

print("\n[Benchmark 1] Single email send time")

single_times = []
for i in range(3):
    start = time.time()
    try:
        await sender.send_email(
            to_email=recipient,
            subject=f"⚡ Performance Test #{i+1}",
            html_content=f"<p>Performance test email {i+1}</p>"
        )
        elapsed = time.time() - start
        single_times.append(elapsed)
        print(f"  Email {i+1}: {elapsed:.3f}s")
    except Exception as e:
        print(f"  Email {i+1}: Failed - {e}")

if single_times:
    avg_time = sum(single_times) / len(single_times)
    print(f"\n  Average time: {avg_time:.3f}s")
    print(f"  Min: {min(single_times):.3f}s")
    print(f"  Max: {max(single_times):.3f}s")

print("\n" + "=" * 80)
print("✅ Performance test complete")
print("\n📊 Typical Performance:")
print("   - Local SMTP server: < 0.1s")
print("   - Gmail SMTP: 1-3s")
print("   - With retry: up to 15s (if failures occur)")

⚡ Test 7: Performance Benchmark

[Benchmark 1] Single email send time
  Email 1: 3.011s
  Email 2: 2.912s
  Email 3: 3.146s

  Average time: 3.023s
  Min: 2.912s
  Max: 3.146s

✅ Performance test complete

📊 Typical Performance:
   - Local SMTP server: < 0.1s
   - Gmail SMTP: 1-3s
   - With retry: up to 15s (if failures occur)


### 9. Convenience Function 테스트

In [11]:
print("🎯 Test 8: Convenience Functions")
print("=" * 80)

# Test convenience function (automatically creates EmailSender)
print("\n[Test 8-1] send_email() convenience function...")

try:
    success = await send_email(
        to_email=recipient,
        subject="🎯 Convenience Function Test",
        html_content="""
        <html>
        <body style="font-family: Arial; padding: 20px;">
            <h1>🎯 Convenience Function</h1>
            <p>This email was sent using the <code>send_email()</code> convenience function.</p>
            <p>No need to create EmailSender instance manually!</p>
            <pre style="background: #f4f4f4; padding: 10px; border-radius: 5px;">
await send_email(
    to_email="user@example.com",
    subject="Hello",
    html_content="&lt;p&gt;Hello!&lt;/p&gt;"
)
            </pre>
        </body>
        </html>
        """
    )
    
    if success:
        print("✅ Convenience function works!")
    else:
        print("❌ Convenience function failed")
        
except Exception as e:
    print(f"❌ Error: {e}")

print("\n✅ Convenience function test complete")

🎯 Test 8: Convenience Functions

[Test 8-1] send_email() convenience function...
✅ Convenience function works!

✅ Convenience function test complete


## 📊 Summary

### EmailSender 테스트 결과

```
테스트 완료 항목:

✅ 1. 환경 변수 설정 확인
✅ 2. EmailSender 초기화
✅ 3. 단일 HTML 이메일 발송
✅ 4. Multipart 이메일 (HTML + Plain Text)
✅ 5. 완전한 다이제스트 이메일 (with EmailBuilder)
✅ 6. 배치 이메일 발송
✅ 7. 에러 핸들링
✅ 8. 재시도 로직 (Exponential Backoff)
✅ 9. 성능 벤치마크
✅ 10. Convenience Functions
```

### 핵심 기능

- ⚡ **비동기 발송**: `aiosmtplib` 사용
- 🔄 **자동 재시도**: 3회 시도, exponential backoff (2s → 4s → 8s)
- 🔒 **TLS 보안**: STARTTLS 암호화 연결
- 📧 **Multipart 지원**: HTML + Plain Text fallback
- 🛡️ **Circuit Breaker**: max_failures로 배치 발송 중단
- 📊 **상세 로깅**: 성공/실패 추적

### 실제 사용 예시

```python
from app.email.sender import EmailSender
from app.email.builder import EmailBuilder

# 1. 이메일 HTML 생성
builder = EmailBuilder()
html = builder.build_daily_digest(
    user_name="김연구",
    user_email="user@example.com",
    articles=articles,
    daily_limit=5
)

# 2. 이메일 발송
sender = EmailSender()
await sender.send_email(
    to_email="user@example.com",
    subject="🔬 오늘의 AI 연구 다이제스트",
    html_content=html
)
```

### 📬 이메일 확인

**받는 사람**: sguys99@gmail.com

총 전송된 이메일:
- Test 1: 간단한 HTML 이메일
- Test 2: Multipart 이메일
- Test 3: 일일 다이제스트 (실제 콘텐츠)
- Test 4: 배치 테스트 (3개)
- Test 6: 재시도 테스트
- Test 7: 성능 테스트 (3개)
- Test 8: Convenience function

**합계: 약 10-12개 이메일**

⚠️ 스팸 폴더도 확인해주세요!

---

**All tests completed! 🎉**